# Ambiente Blackjack

## Objetivo
Este notebook implementa o ambiente do jogo Blackjack que será utilizado nas etapas de simulação, comparação de políticas e treinamento com Q-Learning.

O foco desta etapa é modelar corretamente:
- as regras do jogo;
- a representação do estado;
- a lógica de compra de cartas;
- o cálculo das recompensas;
- a geração de episódios para análise posterior.

## Regras adotadas no ambiente

Neste projeto, o Blackjack foi modelado com as seguintes regras:

- As cartas possíveis são: 1, 2, 3, ..., 10, 10, 10, 10.
- O Ás pode valer 1 ou 11, dependendo da situação mais vantajosa sem ultrapassar 21.
- O jogador pode escolher entre:
  - `hit`: comprar mais uma carta;
  - `stick`: parar.
- O dealer continua comprando cartas até atingir pelo menos 17 pontos.
- A recompensa final segue esta lógica:
  - vitória: `+1`
  - derrota: `-1`
  - empate: `0`

A representação do estado considera:
- a soma da mão do jogador;
- a carta visível do dealer;
- a presença de Ás utilizável.

In [13]:
import pandas as pd
import random

from utils_io import salvar_df

random.seed(42)

## Implementação do ambiente

A seguir são definidas as funções principais do ambiente:

- `comprar_carta()`: sorteia uma carta do baralho simplificado;
- `valor_mao(mao)`: calcula o valor total da mão considerando a lógica do Ás;
- `tem_as_utilizavel(mao)`: identifica se existe um Ás que pode ser tratado como 11 sem estourar;
- `estado_jogo(mao_jogador, carta_dealer)`: representa o estado observado pelo agente;
- `partida_blackjack()`: executa uma partida completa e retorna as informações finais do episódio.

In [14]:
def comprar_carta():
    #Baralho simplificado do Blackjack
    baralho = [1,2,3,4,5,6,7,8,9,10,10,10,10]
    return random.choice(baralho)

def valor_mao(mao):
    total = sum(mao)
    ases = mao.count(1)
    
    while ases > 0 and total + 10 <= 21:
        total += 10
        ases -= 1
        
    return total

def tem_as_utilizavel(mao):
    total = sum(mao)
    return 1 in mao and total + 10 <= 21

def estado_jogo(mao_jogador, carta_dealer):
    return (
        valor_mao(mao_jogador),
        carta_dealer,
        int(tem_as_utilizavel(mao_jogador))
    )

def partida_blackjack(policy=None, verbose=False):
    mao_jogador = [comprar_carta(), comprar_carta()]
    mao_dealer = [comprar_carta(), comprar_carta()]
    
    while True:
        estado = estado_jogo(mao_jogador, mao_dealer[0])
        
        if valor_mao(mao_jogador) >= 21:
            break
        
        if policy is None:
            acao = random.choice(["hit", "stick"])
        else:
            acao = policy(estado)
        
        if acao == "hit":
            mao_jogador.append(comprar_carta())
            if valor_mao(mao_jogador) > 21:
                if verbose:
                    print("Jogador estourou:", mao_jogador)
                return {
                    "estado_final": estado,
                    "resultado": "derrota",
                    "recompensa": -1,
                    "mao_jogador": mao_jogador,
                    "mao_dealer": mao_dealer
                }
        else:
            break

    while valor_mao(mao_dealer) < 17:
        mao_dealer.append(comprar_carta())

    total_jogador = valor_mao(mao_jogador)
    total_dealer = valor_mao(mao_dealer)

    if total_dealer > 21 or total_jogador > total_dealer:
        resultado = "vitoria"
        recompensa = 1
    elif total_jogador < total_dealer:
        resultado = "derrota"
        recompensa = -1
    else:
        resultado = "empate"
        recompensa = 0

    if verbose:
        print("Jogador:", mao_jogador, total_jogador)
        print("Dealer:", mao_dealer, total_dealer)
        print("Resultado:", resultado)

    return {
        "estado_final": estado_jogo(mao_jogador, mao_dealer[0]),
        "resultado": resultado,
        "recompensa": recompensa,
        "mao_jogador": mao_jogador,
        "mao_dealer": mao_dealer
    }

## Validação inicial do ambiente

Antes de gerar uma base maior de episódios, são executadas algumas partidas em modo detalhado para verificar se:

- a lógica da compra de cartas está correta;
- o dealer segue a regra esperada;
- os resultados finais fazem sentido;
- a recompensa está sendo atribuída corretamente.

In [15]:
for i in range(5):
    print(partida_blackjack(verbose=True))
    print("-" * 40)

Jogador: [5, 2] 7
Dealer: [10, 3, 10] 23
Resultado: vitoria
{'estado_final': (7, 10, 0), 'resultado': 'vitoria', 'recompensa': 1, 'mao_jogador': [5, 2], 'mao_dealer': [10, 3, 10]}
----------------------------------------
Jogador: [6, 7] 13
Dealer: [10, 10] 20
Resultado: derrota
{'estado_final': (13, 10, 0), 'resultado': 'derrota', 'recompensa': -1, 'mao_jogador': [6, 7], 'mao_dealer': [10, 10]}
----------------------------------------
Jogador: [10, 3, 5] 18
Dealer: [3, 5, 7, 10] 25
Resultado: vitoria
{'estado_final': (18, 3, 0), 'resultado': 'vitoria', 'recompensa': 1, 'mao_jogador': [10, 3, 5], 'mao_dealer': [3, 5, 7, 10]}
----------------------------------------
Jogador: [10, 2] 12
Dealer: [9, 1] 20
Resultado: derrota
{'estado_final': (12, 9, 0), 'resultado': 'derrota', 'recompensa': -1, 'mao_jogador': [10, 2], 'mao_dealer': [9, 1]}
----------------------------------------
Jogador: [1, 2, 1] 14
Dealer: [4, 9, 2, 5] 20
Resultado: derrota
{'estado_final': (14, 4, 1), 'resultado': 'derr

## Geração de episódios iniciais

Após validar o comportamento do ambiente, são simulados múltiplos episódios para gerar uma base inicial de resultados.

Essa base será utilizada para:
- verificar a distribuição dos resultados;
- inspecionar a consistência do ambiente;
- servir como apoio às etapas seguintes do projeto.

In [16]:
resultados = []

for episodio in range(1000):
    r = partida_blackjack()
    resultados.append({
        "id_episodio": episodio,
        "resultado": r["resultado"],
        "recompensa": r["recompensa"],
        "estado_final": str(r["estado_final"]),
        "mao_jogador": str(r["mao_jogador"]),
        "mao_dealer": str(r["mao_dealer"])
    })

df = pd.DataFrame(resultados)
df.head()

,id_episodio,resultado,recompensa,estado_final,mao_jogador,mao_dealer
0,0,vitoria,1,"(20, 8, 0)","[10, 10]","[8, 9]"
1,1,derrota,-1,"(16, 10, 0)","[8, 8, 10]","[10, 10]"
2,2,vitoria,1,"(19, 2, 0)","[3, 10, 4, 2]","[2, 6, 9]"
3,3,vitoria,1,"(14, 2, 0)","[7, 7]","[2, 1, 10, 3, 7]"
4,4,derrota,-1,"(12, 5, 0)","[2, 10]","[5, 4, 7, 1]"


## Estrutura da base gerada

Cada linha representa um episódio completo do jogo, contendo:

- `id_episodio`: identificador do episódio;
- `resultado`: desfecho da partida;
- `recompensa`: retorno final do episódio;
- `estado_final`: estado final observado;
- `mao_jogador`: cartas finais do jogador;
- `mao_dealer`: cartas finais do dealer.

Esses dados ajudam a validar se a implementação do ambiente está coerente com a dinâmica esperada do Blackjack.

## Persistência dos dados

A base inicial é salva em CSV para permitir reutilização nas próximas etapas do projeto, sem necessidade de reexecutar toda a simulação.

In [17]:
salvar_df(df, "blackjack_resultados_iniciais", pasta="dados")

Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_resultados_iniciais.csv


## Leitura inicial dos resultados

A distribuição entre vitórias, derrotas e empates permite verificar se o ambiente está produzindo resultados plausíveis.

Nesta etapa, o objetivo não é obter alta performance, mas sim confirmar que:
- o jogo está sendo simulado corretamente;
- há variedade de resultados;
- a lógica de recompensa está consistente.

Como ainda não existe uma política inteligente, essa base representa apenas o comportamento do ambiente sob a dinâmica implementada.

In [19]:
df.head(10)

,id_episodio,resultado,recompensa,estado_final,mao_jogador,mao_dealer
0,0,vitoria,1,"(20, 8, 0)","[10, 10]","[8, 9]"
1,1,derrota,-1,"(16, 10, 0)","[8, 8, 10]","[10, 10]"
2,2,vitoria,1,"(19, 2, 0)","[3, 10, 4, 2]","[2, 6, 9]"
3,3,vitoria,1,"(14, 2, 0)","[7, 7]","[2, 1, 10, 3, 7]"
4,4,derrota,-1,"(12, 5, 0)","[2, 10]","[5, 4, 7, 1]"
5,5,derrota,-1,"(12, 10, 0)","[8, 4]","[10, 5, 6]"
6,6,vitoria,1,"(18, 4, 0)","[8, 10]","[4, 9, 9]"
7,7,derrota,-1,"(18, 5, 0)","[9, 8, 1, 8]","[5, 10]"
8,8,derrota,-1,"(19, 10, 0)","[10, 9]","[10, 10]"
9,9,vitoria,1,"(13, 6, 0)","[3, 10]","[6, 7, 10]"


In [20]:
df.groupby("resultado").size().reset_index(name="qtd")

,resultado,qtd
0,derrota,640
1,empate,47
2,vitoria,313


## Conclusão

Nesta etapa foi implementado e validado o ambiente Blackjack que servirá de base para as próximas fases do projeto.

Os testes iniciais indicam que:
- a dinâmica do jogo está funcional;
- os estados estão sendo representados corretamente;
- a estrutura de recompensas está consistente;
- os episódios podem ser armazenados e reutilizados em etapas posteriores.

Com o ambiente validado, o próximo passo é comparar políticas simples e treinar um agente com Q-Learning.